# What p-values and power can do for you — and what they can't

Notebook 01 built the machine that stamps **significant / not significant** on a
study. Now run that machine at scale: a thousand studies (or screened patients, or
tested hypotheses). In each one, reality either holds an effect (**condition +**) or
it doesn't (**condition −**), and the machine ends with a verdict: **predicted +**
("significant!") or **predicted −**.

Two famous numbers describe the *procedure*:

| testing language | diagnostic language | the question it answers |
|---|---|---|
| **1 − α** &nbsp; (α = significance level) | **specificity** | *If there is nothing*, how often do we correctly stay quiet? |
| **power** &nbsp; (1 − β) | **sensitivity**, recall | *If there is something*, how often do we catch it? |

Both questions begin with an ***if* about the truth** — and the truth is precisely what you
don't know. The question you actually face after an experiment runs in the opposite direction:

> "My result came out **positive**. How likely is it that the effect is **real**?"

That number is the **PPV** (positive predictive value, a.k.a. **precision**); its mirror image
for negative results is the **NPV**. Neither α nor power can answer it on their own.
Play with the demo below to see why.


In [3]:
# The counting machinery and the two views.
# Pure functions: (base rate, sensitivity, specificity, total) -> HTML / SVG strings.

# condition positive = blue, condition negative = orange; pale = misclassified
C = {"TP": "#3D6FB0", "FN": "#B6CCE6", "FP": "#F2CDA2", "TN": "#E0883A"}
TINT = {"TP": "#E7EEF7", "FN": "#F4F8FB", "FP": "#FCF4E8", "TN": "#FAEBDB"}
INK, MUTED, RULE = "#333333", "#888888", "#DDDDDD"
FONT = "font-family:system-ui,-apple-system,sans-serif;"

ORDER = ["TP", "FN", "FP", "TN"]
NAMES = {"TP": "true positives", "FN": "false negatives",
         "FP": "false positives", "TN": "true negatives"}


def split(total, frac):
    """Split an integer total into (round(total*frac), remainder)."""
    a = round(total * frac)
    return a, total - a


def confusion(prev, sens, spec, total):
    P, N = split(total, prev)
    TP, FN = split(P, sens)
    TN, FP = split(N, spec)
    return dict(P=P, N=N, TP=TP, FN=FN, FP=FP, TN=TN, PP=TP + FP, PN=FN + TN)


def pct(num, den):
    return f"{num / den:.1%}" if den else "–"


def fmt(n):
    return f"{n:,}"


# ----------------------------------------------------------------------------
# 1 · the 2x2 contingency table
# ----------------------------------------------------------------------------

def table_html(prev, sens, spec, total):
    c = confusion(prev, sens, spec, total)
    td = f"padding:7px 18px;text-align:center;{FONT}font-size:13px;"

    def count(key, sub):
        return (f"<td style='{td}background:{TINT[key]};border:1px solid {RULE};'>"
                f"<span style='font-size:17px;font-weight:600;color:{INK}'>{fmt(c[key])}</span><br>"
                f"<span style='font-size:10px;color:{MUTED};letter-spacing:.5px'>{sub}</span></td>")

    def head(txt):
        return f"<td style='{td}color:{MUTED};font-size:11px'>{txt}</td>"

    def label(txt, color):
        return (f"<td style='{td}text-align:right;color:{color};"
                f"font-weight:600;font-size:12px'>{txt}</td>")

    def total_cell(v):
        return f"<td style='{td}color:{MUTED}'>{fmt(v)}</td>"

    def metric(name, value, strong=False):
        w = "font-weight:700;" if strong else "font-weight:600;"
        return (f"<td style='{td}'><span style='font-size:10px;color:{MUTED};"
                f"letter-spacing:.5px'>{name}</span><br>"
                f"<span style='font-size:15px;{w}color:{INK}'>{value}</span></td>")

    empty = "<td></td>"
    rows = [
        empty + head("predicted +") + head("predicted −") + head("total") + empty,
        label("condition + &nbsp;P", C["TP"]) + count("TP", "TP") + count("FN", "FN")
        + total_cell(c["P"]) + metric("SENSITIVITY (power)", pct(c["TP"], c["P"])),
        label("condition − &nbsp;N", C["TN"]) + count("FP", "FP") + count("TN", "TN")
        + total_cell(c["N"]) + metric("SPECIFICITY (1−α)", pct(c["TN"], c["N"])),
        head("total") + total_cell(c["PP"]) + total_cell(c["PN"]) + total_cell(total) + empty,
        empty + metric("PPV (precision)", pct(c["TP"], c["PP"]), strong=True)
        + metric("NPV", pct(c["TN"], c["PN"]), strong=True) + empty + empty,
    ]
    body = "".join(f"<tr>{r}</tr>" for r in rows)
    return f"<table style='border-collapse:collapse;margin:8px 0'>{body}</table>"


# ----------------------------------------------------------------------------
# 2 · the garden of forked paths — flow thickness carries the counts.
#     Tails (left) grouped by condition; heads (right) grouped by prediction.
# ----------------------------------------------------------------------------

# right-hand stacking: the two predicted-positive outcomes on top, then the two
# predicted-negative ones — so the heads read as the prediction the test made.
HEAD_ORDER = ["TP", "FP", "FN", "TN"]
PRED = {"+": ["TP", "FP"], "−": ["FN", "TN"]}


def swatch_label(x, yc, k, n):
    """Legend row: colour swatch + code/count + outcome name, centred on yc."""
    return (f"<rect x='{x}' y='{yc - 4.5:.1f}' width='9' height='9' fill='{C[k]}'/>"
            f"<text x='{x + 16}' y='{yc - 1:.1f}' style='{FONT}' font-size='12' fill='{INK}'>"
            f"<tspan font-weight='600'>{k}</tspan>"
            f"<tspan font-weight='600'> · {fmt(n)}</tspan></text>"
            f"<text x='{x + 16}' y='{yc + 11:.1f}' style='{FONT}' font-size='10' "
            f"fill='{MUTED}'>{NAMES[k]}</text>")


def ribbon_svg(prev, sens, spec, total):
    c = confusion(prev, sens, spec, total)
    Hin, gap, mt = 330, 16, 26
    gw, gb = 12, 46                          # within-group gap, between-group gap
    xl, x1 = 150, 440                        # ribbons run from xl to the arrow tips
    width = 780

    h = {k: c[k] / total * Hin for k in HEAD_ORDER}

    # left tails — grouped by condition (P: TP,FN  /  N: FP,TN), as before
    yPtop = mt + gap
    yl = {"TP": yPtop + h["TP"] / 2, "FN": yPtop + h["TP"] + h["FN"] / 2}
    yNtop = yPtop + h["TP"] + h["FN"] + gap
    yl["FP"] = yNtop + h["FP"] / 2
    yl["TN"] = yNtop + h["FP"] + h["TN"] / 2
    left_bottom = yNtop + h["FP"] + h["TN"]

    # right heads — grouped by prediction (+: TP,FP  /  −: FN,TN)
    yc_R, zone = {}, {}
    y = mt
    for grp in ("+", "−"):
        top = y
        for j, k in enumerate(PRED[grp]):
            yc_R[k] = y + h[k] / 2
            y += h[k] + (gw if j == 0 else 0)
        zone[grp] = (top, y)
        y += gb
    right_bottom = zone["−"][1]

    height = max(left_bottom, right_bottom) + mt
    xm = (xl + x1) / 2
    parts = []

    for k in HEAD_ORDER:
        t = h[k]
        if t < 0.4:
            continue
        yr = yc_R[k]
        ahead = min(40, max(10, t * 0.35))      # arrowhead grows with the ribbon
        over = min(12, max(2, t * 0.15))
        xe = x1 - ahead
        parts.append(f"<path d='M{xl},{yl[k]:.1f} C{xm},{yl[k]:.1f} {xm},{yr:.1f} {xe},{yr:.1f}' "
                     f"stroke='{C[k]}' stroke-width='{t:.1f}' fill='none'/>")
        parts.append(f"<polygon points='{xe},{yr - t / 2 - over:.1f} {xe},{yr + t / 2 + over:.1f} "
                     f"{x1 + 2},{yr:.1f}' fill='{C[k]}'/>")

    # left source labels — condition + / condition −
    def left_label(yc, color, line1, line2):
        return (f"<text x='{xl - 12}' y='{yc - 2:.1f}' text-anchor='end' style='{FONT}' "
                f"font-size='12' font-weight='600' fill='{color}'>{line1}</text>"
                f"<text x='{xl - 12}' y='{yc + 11:.1f}' text-anchor='end' style='{FONT}' "
                f"font-size='10' fill='{MUTED}'>{line2}</text>")

    hP, hN = h["TP"] + h["FN"], h["FP"] + h["TN"]
    if c["P"]:
        parts.append(left_label(yPtop + hP / 2, C["TP"], f"P · {fmt(c['P'])}", "condition +"))
    if c["N"]:
        parts.append(left_label(yNtop + hN / 2, C["TN"], f"N · {fmt(c['N'])}", "condition −"))

    # right prediction-zone brackets + labels, just past the arrow tips
    xbr = x1 + 16
    for grp, sym in (("+", "predicted +"), ("−", "predicted −")):
        top, bot = zone[grp]
        bot -= (gw if len(PRED[grp]) > 1 else 0)            # trim trailing within-gap
        parts.append(f"<path d='M{xbr},{top:.1f} h6 V{bot:.1f} h-6' fill='none' "
                     f"stroke='{MUTED}' stroke-width='1.2'/>")
        parts.append(f"<text x='{xbr + 12}' y='{(top + bot) / 2 + 4:.1f}' style='{FONT}' "
                     f"font-size='12.5' font-weight='600' fill='{INK}'>{sym}</text>")

    # legend, moved well to the right so nothing crowds the zone labels
    xleg = x1 + 175
    for k in HEAD_ORDER:
        if c[k]:
            parts.append(swatch_label(xleg, yc_R[k], k, c[k]))

    return (f"<svg width='{width}' height='{height:.0f}' "
            f"viewBox='0 0 {width} {height:.0f}' xmlns='http://www.w3.org/2000/svg'>"
            + "".join(parts) + "</svg>")

In [4]:
import ipywidgets as W

def pct_slider(value, mn, mx, step, desc, fmt=".0%"):
    return W.FloatSlider(value=value, min=mn, max=mx, step=step, description=desc,
                         readout_format=fmt, continuous_update=True,
                         style={"description_width": "170px"},
                         layout=W.Layout(width="430px"))

prev_s = pct_slider(0.50, 0.01, 0.99, 0.010, "base rate · P/(P+N)")
sens_s = pct_slider(0.80, 0.05, 1.00, 0.010, "sensitivity · power")
spec_s = pct_slider(0.95, 0.50, 1.00, 0.005, "specificity · 1−α", fmt=".1%")
total_s = W.SelectionSlider(options=[("1,000", 1_000), ("10,000", 10_000),
                                     ("100,000", 100_000), ("1,000,000", 1_000_000)],
                            value=1_000, description="total cases · P+N",
                            continuous_update=True,
                            style={"description_width": "170px"},
                            layout=W.Layout(width="430px"))

table_w, ribbon_w = W.HTML(), W.HTML()

def refresh(change=None):
    args = (prev_s.value, sens_s.value, spec_s.value, total_s.value)
    table_w.value  = table_html(*args)
    ribbon_w.value = ribbon_svg(*args)

for s in (prev_s, sens_s, spec_s, total_s):
    s.observe(refresh, "value")
refresh()

def caption(txt):
    return W.HTML(f"<div style='{FONT}font-size:11px;letter-spacing:1.5px;"
                  f"color:{MUTED};text-transform:uppercase;margin:22px 0 2px'>{txt}</div>")

hint = W.HTML(f"<div style='{FONT}font-size:12px;color:{MUTED};font-style:italic;"
              "margin-bottom:4px'>drag a slider — both views below follow</div>")

W.VBox([hint, prev_s, sens_s, spec_s, total_s,
        caption("1 · the 2×2 table"), table_w,
        caption("2 · the garden of forked paths"), ribbon_w])


## Try this

1. **Defaults** (α = 0.05, power = 0.80, base rate 50 %): PPV ≈ 94 %. Positive results can
   be trusted — this is the comfortable world the textbook numbers implicitly assume.
2. **Drag the base rate down to 10 %** — you are now testing riskier, more surprising
   hypotheses. α and power have not moved, yet PPV falls to ≈ 64 %: one positive in three
   is false.
3. **Down to 2 %** — a long-shot screen. PPV ≈ 25 %: three "discoveries" out of four are
   wrong, with the very same respectable test.
4. Try to rescue it **without touching the base rate**. Pushing power to 100 % barely
   helps (PPV ≈ 29 %). What works is specificity: push it to 99.5 % (α = 0.005) and watch
   the false positives melt away.
5. Back at base rate 2 %, **crank the total cases** from 1,000 to 1,000,000. Every count
   grows a thousandfold — and not a single percentage moves. PPV is a property of
   *proportions*; more cases of the same mix only estimate the same disappointment more
   precisely. **Sample size will not save you.**

### The moral

* **Sensitivity (power)** and **specificity (1 − α)** are properties of the **test**.
  They condition on the truth: *if the effect is real…*, *if it isn't…*
* **PPV** and **NPV** are properties of **your result**: they condition on what you
  actually observed — and they depend, inescapably, on the **base rate**.
* Same test, different population → a positive means something different. p-value and
  power are blind to this; that is not a flaw to fix but a question they were never
  designed to answer. And more data won't fix it either.
* Getting from (sensitivity, specificity, base rate) to (PPV, NPV) is exactly
  **Bayes' theorem** — where we go next.
